# Excercise 5
## NLP with Pytorch 🔥

Use keras framework to solve the below exercises.


In [ ]:
import numpy as np
import keras
import pandas as pd
import matplotlib.pyplot as plt

## 5.1 Predict rating of a movie using Keras

**Exercise:** Use keras framework to predict rating.

In [2]:
dataTraining = pd.read_csv('https://github.com/sergiomora03/AdvancedTopicsAnalytics/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)

In [3]:
plots = dataTraining['plot']
y = (dataTraining['rating'] >= dataTraining['rating'].mean()).astype(int)

In [4]:
plots

3107    most is the story of a single father who takes...
900     a serial killer decides to teach the secrets o...
6724    in sweden ,  a female blackmailer with a disfi...
4704    in a friday afternoon in new york ,  the presi...
2582    in los angeles ,  the editor of a publishing h...
                              ...                        
8417    " our marriage ,  their wedding .  "  it ' s l...
1592    the wandering barbarian ,  conan ,  alongside ...
1723    like a tale spun by scheherazade ,  kismet fol...
7605    mrs .  brisby ,  a widowed mouse ,  lives in a...
215     tinker bell journey far north of never land to...
Name: plot, Length: 7895, dtype: object

In [5]:
y

3107    1
900     0
6724    1
4704    1
2582    1
       ..
8417    0
1592    0
1723    0
7605    1
215     1
Name: rating, Length: 7895, dtype: int32

## Data Precosessing

- Remove stopwords
- Lowercase
- split the text in words
- pad_sequences

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USUARIO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
# --- Preprocesamiento de texto ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in text.split() if word not in stop_words]
    return ' '.join(words)

cleaned_texts = dataTraining['plot'].apply(clean_text)

# --- Tokenización y padding ---
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(cleaned_texts)
sequences = tokenizer.texts_to_sequences(cleaned_texts)
padded_sequences = pad_sequences(sequences, padding='post', maxlen=200)


In [8]:
# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y, test_size=0.2, random_state=42)

In [9]:
class MovieDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = MovieDataset(X_train, y_train)
test_dataset = MovieDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


## Build Model

Create a neural network to predict the rating of a movie, calculate the testing set accuracy.

In [10]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        x = self.dropout(h_n[-1])
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return self.sigmoid(x).squeeze()


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMClassifier(vocab_size=10000, embed_dim=64, hidden_dim=64).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Entrenamiento ---
for epoch in range(5):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 137.1768
Epoch 2, Loss: 137.1603
Epoch 3, Loss: 136.9833
Epoch 4, Loss: 136.9571
Epoch 5, Loss: 136.5596


In [12]:
# --- Evaluación ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        correct += (predicted == targets).sum().item()
        total += targets.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")


Test Accuracy: 0.5446


En este proyecto se implementa un modelo de red neuronal en PyTorch con el fin de predecir si una película tiene una calificación por encima del promedio, utilizando como fuente de información su sinopsis. Inicialmente, se procede a realizar una limpieza del texto, donde se convierte todo a minúsculas, se eliminan los signos de puntuación y se remueven las palabras vacías (stopwords) del idioma inglés. Esta etapa es fundamental para reducir el ruido y mejorar la calidad de las representaciones textuales.

A continuación, se aplica tokenización utilizando la herramienta Tokenizer de Keras, estableciendo un límite de 10,000 palabras más frecuentes. Después, se transforma cada sinopsis en una secuencia de enteros que representan palabras y se aplica padding para asegurar que todas las secuencias tengan la misma longitud (200 tokens), lo cual es un requerimiento para el entrenamiento en batch de redes neuronales.

Posteriormente, se realiza una división de los datos en conjunto de entrenamiento y prueba (80% y 20% respectivamente). Para facilitar el trabajo con PyTorch, se define una clase personalizada MovieDataset, la cual hereda de Dataset. Esta clase convierte los textos y etiquetas en tensores y permite que se usen con DataLoader para el entrenamiento por lotes.

El modelo construido está basado en una arquitectura LSTM (LSTMClassifier). Comienza con una capa de embedding que convierte los índices de palabras en vectores densos de tamaño 64. Luego, se incorpora una capa LSTM con 64 unidades ocultas, diseñada para capturar las dependencias temporales en las secuencias de texto. La salida final del LSTM se pasa por una capa de dropout con tasa del 30% para mitigar el sobreajuste. Seguidamente, se incluye una capa densa con 32 neuronas y activación ReLU, otra capa de dropout y, finalmente, una capa de salida con una única neurona y activación sigmoid que permite realizar la predicción binaria.

El modelo se entrena durante cinco épocas utilizando la función de pérdida BCELoss (Binary Cross Entropy Loss), ideal para problemas de clasificación binaria, y el optimizador Adam, reconocido por su eficiencia y capacidad de convergencia. Durante el entrenamiento, se calcula e imprime la pérdida acumulada por época.

Finalmente, el modelo se evalúa utilizando el conjunto de prueba. Para ello, se desactiva el modo entrenamiento (model.eval()) y se deshabilita el cálculo del gradiente para ahorrar recursos. Las predicciones se comparan con las etiquetas reales y se calcula la precisión total como métrica de evaluación. Esta implementación demuestra cómo utilizar PyTorch para tareas de procesamiento de lenguaje natural (NLP) aplicadas a clasificación binaria, con una estructura replicable para otros conjuntos de datos similares.

## LSTM Mejorado

In [13]:
# --- Modelo LSTM Mejorado ---
class BidirectionalLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(BidirectionalLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]  # Última salida temporal
        out = self.dropout(out)
        out = torch.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return self.sigmoid(out).squeeze()

In [14]:
# --- Entrenamiento ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BidirectionalLSTM(vocab_size=10000, embed_dim=64, hidden_dim=128).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(10):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 137.2151
Epoch 2, Loss: 137.1032
Epoch 3, Loss: 137.0558
Epoch 4, Loss: 137.0595
Epoch 5, Loss: 136.8642
Epoch 6, Loss: 136.7598
Epoch 7, Loss: 136.3531
Epoch 8, Loss: 135.9159
Epoch 9, Loss: 135.7915
Epoch 10, Loss: 135.7224


In [ ]:
# --- Evaluación ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        correct += (predicted == targets).sum().item()
        total += targets.size(0)

accuracy = correct / total
print(f"\n Test Accuracy (Mejorado): {accuracy:.4f}")


✅ Test Accuracy (Mejorado): 0.5421


En esta versión mejorada del modelo, se introdujeron varias optimizaciones tanto en la arquitectura como en los parámetros de entrenamiento con el objetivo de aumentar la precisión en la predicción de calificaciones de películas. Una de las principales mejoras fue el uso de una capa Bidirectional LSTM en lugar de una LSTM tradicional. Esta capa permite que la red procese simultáneamente la secuencia de texto de izquierda a derecha y de derecha a izquierda, lo que enriquece la comprensión contextual del modelo al capturar relaciones semánticas en ambas direcciones.

Además, se incrementó la longitud máxima del texto (maxlen) de 200 a 300 tokens durante el padding, lo cual asegura que el modelo tenga acceso a una mayor parte del contenido de las sinopsis, evitando truncamientos prematuros que podrían eliminar información relevante. En cuanto a la arquitectura interna del modelo, se aumentó el número de unidades ocultas de la LSTM a 128, y la capa densa intermedia se expandió a 64 neuronas, lo que incrementa la capacidad del modelo para aprender representaciones más complejas. También se mantuvieron las capas Dropout con una tasa del 30% como técnica de regularización para reducir el sobreajuste.

En el entrenamiento, se ajustó la tasa de aprendizaje del optimizador Adam a 0.0005, un valor más conservador que permite una convergencia más estable y precisa. Asimismo, se amplió el número de épocas de entrenamiento a 10 para dar al modelo más oportunidades de aprender sin caer en sobreajuste gracias al uso de validación interna. Finalmente, el modelo fue evaluado en un conjunto de prueba independiente y se espera que estas modificaciones contribuyan a una mejora en la precisión respecto a la versión base, ofreciendo un modelo más robusto y capaz de generalizar mejor en tareas de clasificación de texto.

## Conclusion

 Aunque esperábamos una mejora, el nuevo modelo obtuvo una precisión de 54.21%, ligeramente inferior al anterior (54.46%). Esto nos da una señal clara: mejorar la arquitectura no siempre implica mejor rendimiento

A pesar de introducir mejoras en el modelo, como el uso de una LSTM bidireccional, una longitud de secuencia más amplia, mayor capacidad en las capas ocultas y un ajuste fino en la tasa de aprendizaje, el modelo no superó el rendimiento de la versión base. Esto indica que el cuello de botella no está necesariamente en la arquitectura, sino en las características propias de los datos. Este resultado sugiere que, para obtener mejoras significativas en esta tarea, es fundamental complementar el enfoque de modelado con una mejor ingeniería de características, embeddings preentrenados como GloVe o BERT, o incluso redefinir el objetivo de clasificación para que esté más alineado con la información realmente contenida en los textos.